In [1]:
import torch

def rk4_step(f,x,t,dt,*args):
    k1 = f(x,t,*args)
    k2 = f(x+0.5*dt*k1,t+0.5*dt,*args)
    k3 = f(x+0.5*dt*k2,t+0.5*dt,*args)
    k4 = f(x+dt*k3,t+dt,*args)
    return x + dt*(k1 + 2*k2 + 2*k3 + k4)/6

def cgz(u,c,N,k):
    return k*(u - c)**2 / (N)

def G(u,z,H,cns,N,k,alpha,Fl):
    return -torch.sum(torch.stack([ dFdz(Fl,cn,cgz(u,cn,N,k),alpha,z) for cn in cns]),dim=0)

def F(Fl,c,cg,alpha,z):
    return Fl*torch.sign(c)*torch.exp(-torch.cat([torch.zeros(1), torch.cumulative_trapezoid( alpha/cg,x=z)]))

def dFdz(Fl,c,cg,alpha,z):
    F1 = F(Fl,c,cg,alpha,z)
    return F1*(-alpha/cg)


def qbo_1d(u,t,alpha,kappa,w,H,cns,Fl,N,k,z,dz,eps):
    eta = torch.randn_like(u)*eps
    dudz = torch.zeros_like(u) 
    dudz[0] = u[0]/dz
    dudz[1:]= (u[1:]- u[:-1])/dz# first order upwinding 
    dudz[-1] = 0 
    d2udz2 = torch.zeros_like(u)
    d2udz2[0] = (u[1] - 2*u[0])/dz**2
    d2udz2[1:-1] = (u[2:] - 2*u[1:-1] + u[:-2])/dz**2
    d2udz2[-1] = -(u[-1] - u[-2])/dz**2  
    drag = G(u,z,H,cns,N,k,alpha,Fl)
    terms = drag + eta - w*dudz + kappa*d2udz2
    return terms


def forward(alpha,kappa,cn_ptv,cn_neg,Fl,N):
    dz = 0.05
    dt = 0.01
    eps = 1
    L = 3
    T =  250
    w = 0 
    H = 1 
    k = 1
    cns = torch.as_tensor((cn_ptv,cn_neg))
    z = torch.arange(0,L,dz)
    ts = torch.arange(0,T,dt)
    u = torch.zeros((len(ts),len(z)))
    u[0] = torch.zeros(len(z))
    for i,t in enumerate(ts[:-1]):
        u[i+1] = rk4_step(qbo_1d,u[i],t,dt,torch.as_tensor(alpha),torch.as_tensor(kappa),w,H,cns,Fl,N,k,z,dz,eps)
    freqs = torch.fft.fftfreq(len(ts),d=dt) 
    periods = 1/freqs 
    u_qtr = u[:,int(0.25/dz)]
    u_half = u[:,int(0.5/dz)]
    powers_half= torch.abs(torch.fft.fft(u_half))
    period_half = periods[powers_half.argmax().item()]
    amplitude_half= u_half.max() - u_half.min()
    u_rms = torch.sqrt(torch.mean(u_half**2))
    crest_factor = torch.abs(u.max())/u_rms
    return  torch.stack([period_half,amplitude_half,u_rms,crest_factor],dim=0)

In [46]:
alpha = 1 
kappa = 0.05
cn_ptv = 1
cn_ntv = -1 
Fl = 0.5 
N = 1


samples = torch.stack([forward(alpha,kappa,cn_ptv,cn_ntv,Fl,N) for _ in range(50)],dim=0) 


In [ ]:
from abracalibra.forward_model import ForwardModel
from abracalibra.eki import EnsembleKalmanInversion

fwd_3 = ForwardModel(forward,vectorized=False,cn_ptv=1,cn_ntv=-1,N=1)

eki = EnsembleKalmanInversion()